# Laboratory 1: Building a Transformer


This laboratory covers:

* How the attention mechanism uses query, key, and value to assign weights to elements in a sequence   
* Encoder-only, decoder-only, and encoder-decoder Transformers  
* Building a Transformer from scratch for language translation  
* Understanding word embedding and positional encoding   
* Training a Transformer from scratch to translate German to English  

Understanding attention and Transformers is crucial for text-to-image generation for two important reasons.

First, one of the two main methods of text-to-image generation is based on Transformers explicitly. As we will see later, OpenAI's DALL-E treats text-to-image generation as a sequence-prediction problem with a Transformer.  Specifically, an image is broken into multiple patches (for example, into a $16$ by $16$ grid, hence $16\times 16=256$ patches). These patches are then ordered into a sequence: the top left patch first, the patch to the right next, and so on. Based on a text prompt, the trained Transformer in DALL-E predicts the first patch in the image. In the second iteration, the Transformer predicts the second patch based on the text prompt and the first patch. In the third iteration, the Transformer predicts the third patch based on the text prompt and the first two patches. We repeat the process until all $256$ patches in the images are generated. Therefore, in order to understand this first method, we need to have a deep understanding of how attention and Transformers work.

Second, even though the second method of text-to-image generation, diffusion models, is not explicitly based on Transformers, attention and Transformers are working behind the scenes. The attention mechanism and Transformers are crucial to understanding diffusion models because they provide the foundational architecture and computational framework that enable the model's advanced capabilities. The attention mechanism allows the denoising U-Net model (a key component of any diffusion models) to selectively focus on different parts of the input data when generating output. This selective focus is vital for capturing relevant information and ignoring irrelevant parts, leading to better context understanding and more accurate predictions. More importantly, when generating images from text prompts, diffusion models rely on a multi-modal Transformer: contrastive language image pre-training (CLIP). Specifically, CLIP encodes both text descriptions and images into a shared latent space, which is crucial for diffusion models to generate images that are not only high in quality but also closely aligned with the textual descriptions.

For these two reasons, we start with coding the attention mechanism and the Transformer architecture from scratch. In this laboratory, we will learn to implement the attention mechanism and a Transformer based on the paper *Attention Is All You Need*, which first proposed the Transformer architecture. The Transformer, once trained, can handle translations between any two languages (such as German to English or English to Chinese). In particular, we will explore the inner workings of the attention mechanism, including the roles of query, key, and value vectors, and the computation of scaled dot product attention (SDPA). We will construct an encoder layer by integrating layer normalization and residual connection into a multi-head attention layer and combining it with a feed-forward layer. We will then stack six of these encoder layers to form the encoder. Similarly, we will develop a decoder in the Transformer that is capable of generating translation one token at a time, based on previous tokens in the translation and the encoder's output. This groundwork will equip us with the knowledge to train a Transformer for translations between any two languages. We will then train the Transformer using a dataset containing over $29,000$ German-to-English translations. We will witness the trained model translating common German phrases to English with an accuracy comparable to using Google Translate.

# 1. Word Embedding and Positional Encoding
To make the explanation of self-attention and Transformers more relatable, we will use the German to English translation as our example. By working through the example of building and training a model to translate German phrases to English, we will have a deep understanding of various components of a Transformer and how the attention mechanism works.

Imagine that we have collected $29,000$ pairs of German to English translations. Our goal is to create a machine learning model and train the model by using the dataset.

Run the following code cell to install the needed libraries for this laboratory:

In [1]:
pip install spacy

## 1.1. Word Tokenization with the Spacy Library
First we need to download the training dataset. The following code block downloads the file `training.tar.gz` and places it in the folder `files`.

In [2]:
import requests, os, tarfile

url=("https://raw.githubusercontent.com/neychev/"
     "small_DL_repo/master/datasets/Multi30k/training.tar.gz")    #The URL to download the training dataset
os.makedirs("files", exist_ok=True)
if not os.path.exists("files/training.tar.gz"):    #Downloads the dataset
    fb1=requests.get(url)
    with open("files/training.tar.gz","wb") as f:
        f.write(fb1.content)
train=tarfile.open('files/training.tar.gz')    #Unzips the file
train.extractall('files')    #Places content in the files folder
train.close()

/tmp/ipykernel_1029/3577882730.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  train.extractall('files')    #Places content in the files folder


After execution, two files, `train.en` and `train.de` will be saved in the `files` folder. We then read the content of the two files into two lists:

In [3]:
with open("files/train.de", 'rb') as fb:
    trainde = fb.readlines()
with open("files/train.en", 'rb') as fb:
    trainen = fb.readlines()
trainde=[i.decode("utf-8").strip() for i in trainde]
trainen=[i.decode("utf-8").strip() for i in trainen]

The file `train.de` contains $29,001$ German phrases, while `train.en` contains the corresponding English translations. The above code cell reads these phrases and places them in two lists, `trainde` and `trainen`, respectively. We can check the length of each list and print out the first five elements in each list as follows:

In [4]:
from pprint import pprint

print(f"the length of the list trainde is {len(trainde)}")
print(f"the length of the list trainen is {len(trainen)}")
print("the first five elements of the list trainde are")
pprint(trainde[:5])
print("the first five elements of the list trainen are")
pprint(trainen[:5])

the length of the list trainde is 29001
the length of the list trainen is 29001
the first five elements of the list trainde are
['Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.',
 'Ein kleines Mädchen klettert in ein Spielhaus aus Holz.',
 'Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.',
 'Zwei Männer stehen am Herd und bereiten Essen zu.']
the first five elements of the list trainen are
['Two young, White males are outside near many bushes.',
 'Several men in hard hats are operating a giant pulley system.',
 'A little girl climbing into a wooden playhouse.',
 'A man in a blue shirt is standing on a ladder cleaning a window.',
 'Two men are at the stove preparing food.']


Both lists have $29,001$ phrases. The above output shows five German phrases, followed by their English translations.

We will use the tokenizers in the `Spacy` library to convert both German and English phrases into tokens. First, let's define the two tokenizers as follows:

In [5]:
import os, spacy

try:
    de_tokenizer = spacy.load("de_core_news_sm") #Tries to load the German model
except IOError:
    os.system("python -m spacy download de_core_news_sm") #If the German model isn’t found, downloads it
    de_tokenizer = spacy.load("de_core_news_sm")

try:
    en_tokenizer = spacy.load("en_core_web_sm") #Tries to load the English model
except IOError:
    os.system("python -m spacy download en_core_web_sm") # If the English model isn’t found, downloads it
    en_tokenizer = spacy.load("en_core_web_sm")

The models `de_core_news_sm` and `en_core_web_sm` are language-specific pipelines provided by `Spacy`, designed for processing German and English text, respectively. The `sm` indicates that it’s a small, lightweight model, making it faster and less resource-intensive, though it might be less accurate compared to larger models. We name the English tokenizer `en_tokenizer` and the German tokenizer `de_tokenizer`.

Let’s use the two tokenizers to convert the first German phrase and the first English phrase into tokens and print them out:

In [6]:
tokenized_de=[tok.text for tok in
              de_tokenizer.tokenizer(trainde[0])]
tokenized_en=[tok.text for tok in
              en_tokenizer.tokenizer(trainen[0])]
print(tokenized_de)
print(tokenized_en)

['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']


The tokens in the preceding output are either individual words or punctuation. However, it’s possible that some words are broken down into subwords.

Because deep neural networks can’t take tokens as inputs directly, we need to convert tokens into numbers so we can feed them to the Transformer. Next, we build a dictionary for English tokens and map each unique token to a different index.

In [7]:
from collections import Counter

en_tokens=[["BOS"]+[tok.text for tok in en_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainen] #Adds BOS and EOS at the beginning and end of each phrase, respectively
PAD=0
UNK=1
word_count=Counter()
for sentence in en_tokens:
    for word in sentence:
        word_count[word]+=1
frequency=word_count.most_common(50000)
total_en_words=len(frequency)+2
en_word_dict={w[0]:idx+2 for idx,w in enumerate(frequency)} #Assigns an index to each unique token
en_word_dict["PAD"]=PAD
en_word_dict["UNK"]=UNK #The padding token and unknown tokens are assigned indices 0 and 1, respectively
en_idx_dict={v:k for k,v in en_word_dict.items()} #A dictionary to map indices back to tokens

We add tokens `BOS` (beginning of sequence) and `EOS` (end of sequence) to the start and end of each phrase. The dictionary `en_word_dict` maps each unique token to a different index. We also create another dictionary, `en_idx_dict`, to map indices back to their corresponding tokens. This reverse mapping allows us to convert a sequence of indices back into a sequence of tokens to reconstruct the original English phrase. The `PAD` token is used at the end of shorter sequences in a batch so that all sequences are the same length. The `UNK` token is used to represent unknown tokens. This is useful in case we encounter a prompt with unknown tokens, allowing us to convert them to an index to feed to the model.

Using the dictionary `en_word_dict`, we can transform the first English sentence into its numerical representation:

In [8]:
enidx=[en_word_dict.get(i,UNK) for i in tokenized_en]
print(enidx)

[19, 25, 15, 1165, 804, 17, 57, 84, 334, 1329, 5]


We can also convert the sequence of indices back to the original English phrase:

In [9]:
entokens=[en_idx_dict.get(i,"UNK") for i in enidx]
print(entokens)
en_phrase=" ".join(entokens)
for x in '''?:;.,'("-!&)%''':
    en_phrase=en_phrase.replace(f" {x}",f"{x}")
print(en_phrase)

['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']
Two young, White males are outside near many bushes.


Next, we build a dictionary for German tokens and map each unique token to a different index:

In [10]:
de_tokens=[["BOS"]+[tok.text for tok in de_tokenizer.tokenizer(x)]
           +["EOS"] for x in trainde] #Adds BOS and EOS at the beginning and end of each phrase, respectively
de_word_count=Counter()
for sentence in de_tokens:
    for word in sentence:
        de_word_count[word]+=1
defrequency=de_word_count.most_common(50000)
total_de_words=len(defrequency)+2
de_word_dict={w[0]:idx+2 for idx,w in enumerate(defrequency)} #Assigns an index to each unique token
de_word_dict["PAD"]=PAD
de_word_dict["UNK"]=UNK #The padding token and unknown tokens are assigned indices 0 and 1, respectively
de_idx_dict={v:k for k,v in de_word_dict.items()} #A dictionary to map indices back to tokens

The dictionary `de_word_dict` maps each unique German token to an integer, and the dictionary `de_idx_dict` maps integers back to German tokens. Here, we convert the first German phrase to numerical representations:

In [11]:
deidx=[de_word_dict.get(i,UNK) for i in tokenized_de]
print(deidx)

[21, 85, 257, 31, 87, 22, 94, 7, 16, 112, 5497, 3161, 4]


The following code cell converts the preceding numerical representations back to German tokens and restores the original German phrase:

In [12]:
detokens=[de_idx_dict.get(i,"UNK") for i in deidx]
print(detokens)
de_phrase=" ".join(detokens)

for x in '''?:;.,'("-!&)%''':
    de_phrase=de_phrase.replace(f" {x}",f"{x}")
print(de_phrase)

['Zwei', 'junge', 'weiße', 'Männer', 'sind', 'im', 'Freien', 'in', 'der', 'Nähe', 'vieler', 'Büsche', '.']
Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.


Now that we know how to tokenize and index both German and English phrases, we can arrange them into batches to use for training the Transformer later.

## 1.2. A Sequence Padding Function
The numerical representations that we feed to the Transformer must have the same length in each batch. However, the English and German phrases have various lengths. To overcome this, we add $0$s at the end of numerical representations of shorter phrases so all inputs to the Transformer model have the same lengths in each batch.

First, we convert all English phrases to their numerical representations and do the same for German phrases:

In [13]:
out_en_ids=[[en_word_dict.get(w,UNK) for w in s]
            for s in en_tokens]
out_de_ids=[[de_word_dict.get(w,UNK) for w in s]
            for s in de_tokens]
sorted_ids=sorted(range(len(out_de_ids)),
                  key=lambda x:len(out_de_ids[x]))
out_de_ids=[out_de_ids[x] for x in sorted_ids]
out_en_ids=[out_en_ids[x] for x in sorted_ids]

Next, we put the numerical representations into batches for training:

In [14]:
import numpy as np

batch_size=128
idx_list=np.arange(0,len(de_tokens),batch_size)
np.random.shuffle(idx_list)

batch_indexs=[]
for idx in idx_list:
    batch_indexs.append(np.arange(idx,min(len(de_tokens),
                                          idx+batch_size)))

Note that we have sorted observations in the training dataset by the length of the German phrases before placing them into batches. This method ensures that the observations within each batch are of a comparable length, consequently decreasing the need for padding. As a result, this approach not only reduces the overall size of the training data but also accelerates the training process.

To pad sequences, we define the following `seq_padding()` function:

In [15]:
def seq_padding(X, padding=PAD):
    L = [len(x) for x in X]
    ML = max(L)
    padded_seq = np.array([np.concatenate([x,
                   [padding] * (ML - len(x))])
        if len(x) < ML else x for x in X])
    return padded_seq

The function first finds out the longest sequence in the batch and adds $0$s at the end of other sequences so that all sequences in the batch have the same length.

The `Batch` class is defined as follows:

In [16]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import numpy as np
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = np.triu(np.ones(attn_shape),
                              k=1).astype('uint8')
    output = torch.from_numpy(subsequent_mask) == 0
    return output

def make_std_mask(tgt, pad):
    tgt_mask=(tgt != pad).unsqueeze(-2)
    output=tgt_mask & subsequent_mask(\
        tgt.size(-1)).type_as(tgt_mask.data)
    return output

class Batch:
    def __init__(self, src, trg=None, pad=0):
        src = torch.from_numpy(src).to(DEVICE).long()
        self.src = src
        self.src_mask = (src != pad).unsqueeze(-2) #Creates a source mask to hide padding at the end of the sentence
        if trg is not None:
            trg = torch.from_numpy(trg).to(DEVICE).long()
            self.trg = trg[:, :-1] #Creates input to the decoder
            self.trg_y = trg[:, 1:] #Shifts the input one token to the right and uses it as output
            self.trg_mask = make_std_mask(self.trg, pad) #Creates a target mask
            self.ntokens = (self.trg_y != pad).data.sum()

The `Batch` class processes a batch of German and English phrases, converting them
into a format suitable for training. To make this explanation more tangible, consider
the German phrase “Wie geht es dir” and its English equivalent “How are you” as our
example. The `Batch` class receives two inputs: `src`, which is the sequence of indices
representing the tokens in “Wie geht es dir”, and `trg`, the sequence of indices for the
tokens in “How are you”. This class generates a tensor, `src_mask`, to conceal the padding
at the sentence’s end.

The `Batch` class additionally prepares the input and target for the Transformer’s
decoder. The English phrase “How are you” is transformed into five tokens: `[BOS, how, are, you, EOS]`. The first four tokens serve as the input to the decoder, named `trg`.
Next, we shift this input one token to the right to form the target, `trg_y`, which contains
tokens `[how, are, you, EOS]`. Each token in the target sequence is the next token
in the corresponding position in the input sequence. The input sequence is fed to the
decoder to make predictions on what the next token should be. The target sequence
serves as the ground truth. We compare the predicted output from the decoder with
the target sequence and calculate the cross-entropy loss. During training, we modify
model parameters to minimize this loss. This approach compels the model to predict
the next token based on the previous ones.

The `Batch` class also generates a mask, `trg_mask`, for the decoder’s input. The purpose
of this mask is to conceal the subsequent tokens in the input, ensuring that the
model relies solely on previous tokens for making predictions. Text generation (in our
case, generating English translation for German phrases) requires the model to predict
the next token based only on tokens it has seen so far, not future tokens. If future tokens
were visible during training, the model wouldn’t learn the proper dependencies. The
mask ensures that the model’s behavior during training matches its behavior during
inference, leading to better generalization. We can now use the `Batch` class and put the training data in batches:

In [17]:
batches=[]
for b in batch_indexs:
    batch_en=[out_en_ids[x] for x in b]
    batch_de=[out_de_ids[x] for x in b]
    batch_en=seq_padding(batch_en)
    batch_de=seq_padding(batch_de)
    batches.append(Batch(batch_de,batch_en))

The `batches` list contains data batches for training. Each batch has $128$ pairs of numerical representations of German and English phrases,  respectively.

## 1.3. Word Embedding
The number of indices in numerical representations of the German and English phrases
is fairly large. We can find out how many distinct indices we need for each language by
counting the number of elements in the dictionaries `en_word_dict` and `de_word_dict`:

In [18]:
src_vocab = len(de_word_dict)
tgt_vocab = len(en_word_dict)
print(f"there are {src_vocab} distinct German tokens")
print(f"there are {tgt_vocab} distinct English tokens")

there are 19214 distinct German tokens
there are 10837 distinct English tokens


There are $19,214$ distinct German tokens and $10,837$ distinct English tokens in our
dataset. We will use a vector of length `d_model=256`, for example, to represent each token.
The method is called word embedding. The `Embeddings` class is defined as follows:

In [19]:
import math
from torch import nn

class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model

    def forward(self, x):
        out = self.lut(x) * math.sqrt(self.d_model)
        return out

Word embedding transforms tokens into dense, low-dimensional vectors that capture
semantic relationships among them. These embeddings place similar words closer
in vector space, making them more efficient for machine learning models to process
compared to sparse one-hot encodings. Moreover, the embeddings are learned directly
from the training dataset during the model’s training process.

## 1.4. Positional Encoding
To model the order of
elements in the input and output sequences, we will first create positional encodings of
the sequences.

In [20]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000): #Initiates the class, allowing a maximum of 5,000 positions
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model, device=DEVICE)
        position = torch.arange(0., max_len,
                                device=DEVICE).unsqueeze(1)
        div_term = torch.exp(torch.arange(
            0., d_model, 2, device=DEVICE)
            * -(math.log(10000.0) / d_model))
        pe_pos = torch.mul(position, div_term)
        pe[:, 0::2] = torch.sin(pe_pos) #Applies the sine function to even indices in the vector
        pe[:, 1::2] = torch.cos(pe_pos) #Applies the cosine function to odd indices in the vector
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)].requires_grad_(False)
        out = self.dropout(x)
        return out

The purpose of positional encoding is to determine the relative or absolute position of elements in a sequence. Specifically, the positional encoding class we defined above uses the following sine and cosine functions, based on the position $pos$ and the dimension $i$:
$$PE_{(pos,2i)}=\sin\left(\frac{pos}{10000^{2i/d_{model}}}\right),$$
$$PE_{(pos,2i+1)}=\cos\left(\frac{pos}{10000^{2i/d_{model}}}\right).$$
The `PositionalEncoding` class generates vectors for sequence positions using sine
functions for even indices and cosine functions for odd indices. One of the benefits
of using these trigonometric functions is that their outputs range between $–1$ and $1$.
Additionally, it’s important to note that `requires_grad(False)` means there’s no need
to train these values. They remain constant across all inputs, and they don’t change
during training.

# 2. Creating an Encoder-Decoder Transformer
Following the methodology outlined in the *Attention Is All You Need* paper, we will
develop an encoder–decoder Transformer to translate German to English. This section
discusses the process of building the encoder and decoder. The encoder compresses a
German sentence (e.g., “Wie geht es dir”) into a vector representation that captures its
meaning. The decoder generates the English translation, “How are you”, based on the
encoder’s output. We will learn how to construct various sublayers in the encoder and
decoder, including how to implement the attention mechanism.

## 2.1. The Attention Mechanism

The SDPA attention mechanism uses query, key, and value to calculate the relationships
among elements in a sequence. It assigns scores to show how an element in a sequence
is related to all other elements.
The `attention()` function is defined as follows:

In [21]:
def attention(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query,
              key.transpose(-2, -1)) / math.sqrt(d_k) #The scaled attention score is the dot product of query and key, scaled by the square root of d_k
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9) #If there’s a mask, hide future elements in the sequence
    p_attn = nn.functional.softmax(scores, dim=-1) #Calculates attention weights
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn #Returns both attention and attention weights

The `attention()` function takes query, key, and value as inputs and calculates attention
and attention weights. The scaled attention score
is the dot product of query and key, scaled by the square root of the dimension of the
key, $d_k$. We apply the softmax function on the scaled attention score to obtain attention
weights. Finally, attention is calculated as the dot product of attention weights and
value.

Further, instead of using one set of query, key, and value vectors, the Transformer
model uses a concept called multi-head attention. This way, each head captures a different
aspect of the input data so the model can form a more detailed understanding
of the text. Our $256$-dimensional query, key, and value vectors are split into $8$ heads,
and each head has a set of query, key, and value vectors with dimensions of $32$ (because
$256 ÷ 8 = 32$). This is implemented in the `MultiHeadedAttention` class defined below.

In [22]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, h, d_model, dropout=0.1):
        super().__init__()
        assert d_model % h == 0
        self.d_k = d_model // h
        self.h = h
        self.linears = nn.ModuleList([deepcopy(
            nn.Linear(d_model, d_model)) for i in range(4)])
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [l(x).view(nbatches, -1, self.h,
           self.d_k).transpose(1, 2)
         for l, x in zip(self.linears, (query, key, value))]
        x, self.attn = attention(
            query, key, value, mask=mask, dropout=self.dropout)
        x = x.transpose(1, 2).contiguous().view(
            nbatches, -1, self.h * self.d_k)
        output = self.linears[-1](x)
        return output

Each encoder layer and decoder layer also contains a feed-forward sublayer. It doesn’t
treat the sequence of embeddings as a single vector, so we often call it a position-wide
feed-forward network (or a 1D convolutional network). The network is defined in
the `PositionwiseFeedForward` class below.

In [23]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        h1 = self.w_1(x)
        h2 = self.dropout(h1)
        return self.w_2(h2)

## 2.2.	Defining the `Transformer` Class
To create an encoder–decoder Transformer, we define a `Transformer` class as follows:

In [24]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
from torch import nn

class Transformer(nn.Module):
    def __init__(self, encoder, decoder,
                 src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder #Defines an encoder in the transformer
        self.decoder = decoder #Defines a decoder in the transformer
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt),
                            memory, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask) #The source language is encoded into an abstract vector representation by the encoder
        output = self.decode(memory, src_mask, tgt, tgt_mask) #The decoder uses the vector representation to generate the translation in the target language
        return output

The Transformer consists of an encoder and a decoder. The encoder converts the
numerical representation of a German phrase into an abstract representation
(called `memory` in the preceding `Transformer` class). The decoder then takes the output
from the encoder and generates the translation in an autoregressive fashion: it
generates one element at a time, based on the previously generated elements and the
output from the encoder.

The encoder consists of `N=6` identical encoder layers. The `Encoder` class is defined as follows:

In [25]:
from copy import deepcopy

class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)])
        self.norm = LayerNorm(layer.size)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
            output = self.norm(x)
        return output

Inside each encoder layer, there are two sublayers: a multi-head self-attention layer and
a feed-forward layer. We apply layer normalization and residual connection in each sublayer.
The normalization layer normalizes each of the inputs in a batch independently
across all features so that they all have zero mean and unit standard deviation. This
helps stabilize training. Residual connection means the input to each sublayer is added
to the output of the sublayer.

In [26]:
class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(2)])
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](
            x, lambda x: self.self_attn(x, x, x, mask))
        output = self.sublayer[1](x, self.feed_forward)
        return output

class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        output = x + self.dropout(sublayer(self.norm(x)))
        return output

class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        x_zscore = (x - mean) / torch.sqrt(std ** 2 + self.eps)
        output = self.a_2*x_zscore+self.b_2
        return output

The decoder in the Transformer consists of `N = 6` identical decoder layers. Each
decoder layer consists of a multi-head self-attention layer and a basic, position-wise, fully
connected feed-forward network with residual connections. In
addition to these two sublayers, each decoder layer also has a third sublayer that applies
multi-head attention over the encoder stack’s output. Furthermore, the decoder stack’s
self-attention sublayer is masked to prevent positions from attending to subsequent
positions. The mask forces the model to use previous elements in a sequence to predict
later elements.

In [27]:
class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = nn.ModuleList(
            [deepcopy(layer) for i in range(N)])
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        output = self.norm(x)
        return output

Similar to what we did with the encoder layers, we apply residual connection and layer normalization in each decoder layer. The `DecoderLayer` class is defined as follows:

In [28]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn,
                 feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = nn.ModuleList([deepcopy(
        SublayerConnection(size, dropout)) for i in range(3)])

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.sublayer[0](x, lambda x:
                 self.self_attn(x, x, x, tgt_mask)) #The first sublayer is a masked multi-head attention layer
        x = self.sublayer[1](x, lambda x:
                 self.src_attn(x, memory, memory, src_mask)) #The second sublayer is a cross-attention layer between the two languages
        output = self.sublayer[2](x, self.feed_forward) #The third sublayer is a feed-forward network
        return output

Each decoder layer has three sublayers: a multi-head self-attention layer, a multi-head
cross-attention, and a feed-forward network. To calculate the cross-attention in the second
sublayer, we feed the output from the first sublayer, termed `x`, and the output of
the encoder stack, termed `memory`, in the preceding code block, to the `attention()`
function we defined earlier, where `x` is used as query, and `memory` is used as key and value.

The output from the decoder is the probability distribution over all tokens in the
English vocabulary. Because there are $10,837$ different English tokens in our dataset,
when the input to the decoder contains $4$ tokens—for example, `[BOS, how, are, you]`—
the output will have a shape of $4 \times 10,837$. Here, $4$ indicates the number of predicted
tokens, and $10,837$ is the number of elements in the probability distribution.

## 2.3. Creating a Language Translator
We define a `Generator` class to generate the most likely next token.
The output of the `Generator` class is the probability distribution in the target language
vocabulary. This allows the model to predict one token at a time in an autoregressive
fashion, based on previously generated tokens and the output from the encoder.

In [29]:
class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)

    def forward(self, x):
        out = self.proj(x)
        probs = nn.functional.log_softmax(out, dim=-1)
        return probs

Now we are ready to put all the pieces together and create our Transformer model.
The `create_model()` function accomplishes that.

In [30]:
def create_model(src_vocab, tgt_vocab, N, d_model,
                 d_ff, h, dropout=0.1):
    attn=MultiHeadedAttention(h, d_model).to(DEVICE)
    ff=PositionwiseFeedForward(d_model, d_ff, dropout).to(DEVICE)
    pos=PositionalEncoding(d_model, dropout).to(DEVICE)
    model = Transformer(
        Encoder(EncoderLayer(d_model,deepcopy(attn),deepcopy(ff),
                             dropout).to(DEVICE),N).to(DEVICE), #Creates an encoder by instantiating the Encoder class
        Decoder(DecoderLayer(d_model,deepcopy(attn),
             deepcopy(attn),deepcopy(ff), dropout).to(DEVICE),
                N).to(DEVICE), #Creates a decoder by instantiating the Decoder class
        nn.Sequential(Embeddings(d_model, src_vocab).to(DEVICE),
                      deepcopy(pos)), #Creates src_embed by generating input embeddings for the source language
        nn.Sequential(Embeddings(d_model, tgt_vocab).to(DEVICE),
                      deepcopy(pos)), #Creates tgt_embed by generating input embeddings for the target language
        Generator(d_model, tgt_vocab)).to(DEVICE) #Creates a generator by instantiating the Generator class
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model.to(DEVICE)

The `create_model()` function uses the `Transformer` class we defined earlier, with
five essential elements: `encoder`, `decoder`, `src_embed`, `tgt_embed`, and `generator`.
We construct these five components by instantiating `Encoder`, `Decoder`, and
`Generator`
classes. To generate the source language embedding, we process
numerical representations of German phrases using word embedding and positional
encoding, combining the results to form the `src_embed` component. We do
the same for the target language to create `tgt_embed`.

Finally, we use the `create_model()` function and construct a Transformer so that we can employ it to train the German to English translator:

In [31]:
model = create_model(src_vocab, tgt_vocab, N=6,
    d_model=256, d_ff=1024, h=8, dropout=0.1)

The original 2017 paper uses various combinations of hyperparameters when constructing the model. Here we choose a model dimension of $256$ with $8$ heads.

# 3. Training and Using the German to English Translator

Our German to English translator is essentially a multicategory classifier. The objective is
to predict the next token in the English vocabulary when translating a German sentence.

In this section, we will first train the encoder–decoder Transformer using batches of
German-to-English translations, prepared earlier in this laboratory, as our training dataset.
After the model is trained, we will translate German phrases to English.

## 3.1. Loss Function and Optimizer
The original 2017 paper uses label smoothing during training. Label smoothing is a technique used to address overconfidence problems (the predicted probability is greater than the true probability) in classifications.

We define the following class:

In [32]:
class LabelSmoothing(nn.Module):
    def __init__(self, size, padding_idx, smoothing=0.0):
        super().__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.size = size
        self.true_dist = None

    def forward(self, x, target):
        assert x.size(1) == self.size
        true_dist = x.data.clone()
        true_dist.fill_(self.smoothing / (self.size - 2))
        true_dist.scatter_(1,
               target.data.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        mask = torch.nonzero(target.data == self.padding_idx)
        if mask.dim() > 0:
            true_dist.index_fill_(0, mask.squeeze(), 0.0)
        self.true_dist = true_dist
        output = self.criterion(x, true_dist.clone().detach())
        return output

The optimizer we use is the Adam optimizer with $\beta_1=0.9$, $\beta_2=0.98$, and $\epsilon=10^{-9}$. We define the `NoamOpt` class to change the learning rate during the training process:

In [33]:
class NoamOpt:
    def __init__(self, model_size, factor, warmup, optimizer):
        self.optimizer = optimizer
        self._step = 0
        self.warmup = warmup
        self.factor = factor
        self.model_size = model_size
        self._rate = 0

    def step(self):
        self._step += 1
        rate = self.rate()
        for p in self.optimizer.param_groups:
            p['lr'] = rate
        self._rate = rate
        self.optimizer.step()

    def rate(self, step=None):
        if step is None:
            step = self._step
        output = self.factor * (self.model_size ** (-0.5) *
        min(step ** (-0.5), step * self.warmup ** (-1.5)))
        return output

To create the loss function for training, we first define the following class:

In [34]:
class SimpleLossCompute:
    def __init__(self, generator, criterion, opt=None):
        self.generator = generator
        self.criterion = criterion
        self.opt = opt

    def __call__(self, x, y, norm):
        x = self.generator(x)
        loss = self.criterion(x.contiguous().view(-1, x.size(-1)),
                              y.contiguous().view(-1)) / norm
        loss.backward()
        if self.opt is not None:
            self.opt.step()
            self.opt.optimizer.zero_grad()
        return loss.data.item() * norm.float()

We then define the optimizer and loss function as follows:

In [35]:
import torch

optimizer = NoamOpt(256, 1, 2000, torch.optim.Adam(
    model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9))
criterion = LabelSmoothing(tgt_vocab,
                           padding_idx=0, smoothing=0.0)
loss_func = SimpleLossCompute(
            model.generator, criterion, optimizer)

Next, we will train the Transformer by using the data we prepared earlier in the laboratory.

## 3.2. The Training Process
We will train the model for $50$ epochs. We will calculate the loss and the number of tokens from each batch. After each epoch, we calculate the average loss in the epoch as the ratio between the total loss and the total number of tokens:

In [42]:
for epoch in range(5):
    model.train()
    tloss=0
    tokens=0
    for batch in batches:
        out = model(batch.src, batch.trg,
                    batch.src_mask, batch.trg_mask) #Predicts the next token using the transformer
        loss = loss_func(out, batch.trg_y, batch.ntokens) #Calculates loss and adjusts model parameters
        tloss += loss
        tokens += batch.ntokens #Counts the number of tokens in the batch
    print(f"Epoch {epoch}, average loss: {tloss/tokens}")
torch.save(model.state_dict(),"files/de2en.pth") #Saves the weights in the trained model after training

Epoch 0, average loss: 0.7478054165840149
Epoch 1, average loss: 0.7155120372772217
Epoch 2, average loss: 0.6802292466163635
Epoch 3, average loss: 0.6531794667243958
Epoch 4, average loss: 0.6218408346176147


The above training process takes about $20$ minutes if we are using a GPU. It may take several hours if we are using CPU training. Once the training is done, the model weights are saved as `de2en.pth`. Alternatively, it can be downloaded from Google Drive https://drive.google.com/file/d/1fPkiHr8gu2bwKF0Ao7dMwnd8VbNSrGqR/view?usp=sharing. Unzip the file after downloading.

## 3.3. Translating German to English with the Trained Model
Now that we have trained the Transformer, we can use it to translate any German sentence to English. We define a function `de2en()` as follows:

In [50]:
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
def de2en(ger):
    tokenized_ger= [tok.text for tok in de_tokenizer.tokenizer(ger)]
    tokenized_ger=["BOS"]+tokenized_ger+["EOS"]
    geridx=[de_word_dict.get(i,UNK) for i in tokenized_ger]
    src=torch.tensor(geridx).long().to(DEVICE).unsqueeze(0)
    src_mask=(src!=0).unsqueeze(-2)
    memory=model.encode(src,src_mask)    #Uses the encoder to convert German to vector representations
    start_symbol=en_word_dict["BOS"]
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data)
    translation=[]
    for i in range(100):
        out = model.decode(memory,src_mask,ys,
        subsequent_mask(ys.size(1)).type_as(src.data))
        prob = model.generator(out[:, -1])    #Predicts the next English token using the decoder
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        ys = torch.cat([ys, torch.ones(1, 1).type_as(
            src.data).fill_(next_word)], dim=1)
        sym = en_idx_dict[ys[0, -1].item()]
        if sym != 'EOS':    #Stops translating when the next token is EOS
            translation.append(sym)
        else:
            break
    trans=" ".join(translation)
    for x in '''?:;.,'("-!&)%''':
        trans=trans.replace(f" {x}",f"{x}")    #Joins the predicted tokens to form an English sentence as the translation
    return trans

In the `de2en()` function, we first use the tokenizer to convert the German sentence to
tokens. We then add `BOS` and `EOS` to the beginning and the end of the sequence, respectively.
We use the dictionary `de_word_dict` that we created earlier in the laboratory to convert
tokens to indices. The encoder produces a tensor that captures the meaning of the
German sentence and passes it to the decoder.

Based on the output produced by the encoder, the decoder in the trained model
starts translation in an autoregressive manner, starting with the beginning token `BOS`.
In each time step, the decoder generates the most likely next token based on previously
generated tokens, until the predicted token is `EOS`, which signals the end of the
sequence.

Now let’s try to translate five German sentences in the list `trainde`:

In [51]:
model.load_state_dict(torch.load("files/de2en.pth",
            weights_only=True, map_location=DEVICE))
model.eval()
for i in range(5):
    print("original Ger:", trainde[100+i])
    print("original Eng:", trainen[100+i])
    print("translated Eng:", de2en(trainde[100+i]))

original Ger: Männliches Kleinkind in einem roten Hut, das sich an einem Geländer festhält.
original Eng: Toddler boy in a red hat holding on to some railings.
translated Eng: Toddler boy in a red hat holding on to some railings.
original Ger: Drei Hunde stehen auf einer Wiese und eine Person kniet in der Nähe.
original Eng: Three dogs stand in a grassy field while a person kneels nearby.
translated Eng: Three dogs stand in a grassy field and one person near them.
original Ger: Ein Mann steht vor einem Hochhaus.
original Eng: A man is standing in front of a skyscraper
translated Eng: A man is standing in front of a skyscraper
original Ger: Eine Frau fährt ihr Baby in einem Sportwagen im örtlichen Park spazieren.
original Eng: A woman is walking her baby with a stroller at the local park.
translated Eng: A woman is walking her baby in a stroller at the local park.
original Ger: Ein Mann in einem roten Hemd sitzt neben Obst, das zu verkaufen ist.
original Eng: A man in a red shirt is sit

As we can see, the output compares the original English translation and the translation
generated by the trained model. The generated English translations are close to
the original translations.

While the transformer architecture is originally designed for NLP tasks, it can be used
to tackle other challenges in machine learning. In the next laboratory, we will learn to
apply Transformer models to computer vision tasks. Specifically, we will learn to convert
an image into a sequence of tokens, similar to how text is transformed into a sequence
of words. We will then create a Vision Transformer and train it to classify images into different categories.

# Exercises

1.   Print out the last five elements in the lists `trainde` and `trainen`.

In [52]:
from pprint import pprint

print("Last 5 German sentences:")
pprint(trainde[-5:])

print("\nLast 5 English sentences:")
pprint(trainen[-5:])

Last 5 German sentences:
['Ein Bergsteiger übt an einer Kletterwand.',
 'Zwei Bauarbeiter arbeiten auf einer Straße vor einem Hauses.',
 'Ein älterer Mann sitzt mit einem Jungen mit einem Wagen vor einer Fassade.',
 'Ein Mann in Shorts und Hawaiihemd lehnt sich über das Geländer eines '
 'Lotsenboots, mit Nebel und Bergen im Hintergrund.',
 '']

Last 5 English sentences:
['A rock climber practices on a rock climbing wall.',
 "Two male construction workers are working on a street outside someone's home",
 'An elderly man sits outside a storefront accompanied by a young boy with a '
 'cart.',
 'A man in shorts and a Hawaiian shirt leans over the rail of a pilot boat, '
 'with fog and mountains in the background.',
 '']


2.   Convert the second German phrase in `trainde` into a list of tokens, and name the list `tokenized_de1`. Then, convert the second English phrase in `trainen` into a list of tokens, and name the list `tokenized_en1`. Print out the lists `tokenized_de1` and `tokenized_en1`.

In [53]:
tokenized_de1 = [tok.text for tok in de_tokenizer.tokenizer(trainde[1])]
tokenized_en1 = [tok.text for tok in en_tokenizer.tokenizer(trainen[1])]

print("tokenized_de1:", tokenized_de1)
print("tokenized_en1:", tokenized_en1)

tokenized_de1: ['Mehrere', 'Männer', 'mit', 'Schutzhelmen', 'bedienen', 'ein', 'Antriebsradsystem', '.']
tokenized_en1: ['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']


3.   Use the dictionary `en_word_dict` to convert `tokenized_en1` that we created in Exercise 2 into a list of indices `enidx1`. Then, convert `enidx1` back into a list of tokens `entokens1` using the dictionary `en_idx_dict`. Finally, join the tokens in `entokens1` into a sentence.

In [54]:
enidx1 = [en_word_dict.get(tok, UNK) for tok in tokenized_en1]
print("enidx1:", enidx1)

entokens1 = [en_idx_dict.get(idx, "UNK") for idx in enidx1]
print("entokens1:", entokens1)

en_sentence1 = " ".join(entokens1)
for x in '''?:;.,'("-!&)%''':
    en_sentence1 = en_sentence1.replace(f" {x}", f"{x}")
print("Reconstructed English sentence:", en_sentence1)

enidx1: [164, 36, 7, 335, 286, 17, 1208, 2, 753, 3933, 2710, 5]
entokens1: ['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']
Reconstructed English sentence: Several men in hard hats are operating a giant pulley system.


4.   Use the dictionary `de_word_dict` to convert `tokenized_de1` that we created in Exercise 2 into a list of indices `deidx1`. Then, convert `deidx1` back into a list of tokens `detokens1` using the dictionary `de_idx_dict`. Finally, join the tokens in `detokens1` into a sentence.

In [55]:
deidx1 = [de_word_dict.get(tok, UNK) for tok in tokenized_de1]
print("deidx1:", deidx1)

detokens1 = [de_idx_dict.get(idx, "UNK") for idx in deidx1]
print("detokens1:", detokens1)

de_sentence1 = " ".join(detokens1)
for x in '''?:;.,'("-!&)%''':
    de_sentence1 = de_sentence1.replace(f" {x}", f"{x}")
print("Reconstructed German sentence:", de_sentence1)

deidx1: [84, 31, 10, 838, 2096, 15, 8014, 4]
detokens1: ['Mehrere', 'Männer', 'mit', 'Schutzhelmen', 'bedienen', 'ein', 'Antriebsradsystem', '.']
Reconstructed German sentence: Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.


5.   Use the `de2en()` function to translate the 11th and 12th sentences in `trainde` into
English. Compare the translations with the original English translations in `trainen`.

In [56]:
for i in [10, 11]:
    print(f"\nSentence #{i+1}")
    print("Original German   :", trainde[i])
    print("Original English  :", trainen[i])
    print("Translated English:", de2en(trainde[i]))


Sentence #11
Original German   : Eine Ballettklasse mit fünf Mädchen, die nacheinander springen.
Original English  : A ballet class of five girls jumping in sequence.
Translated English: A ballet class of five girls jumping in sequence.

Sentence #12
Original German   : Vier Typen, von denen drei Hüte tragen und einer nicht, springen oben in einem Treppenhaus.
Original English  : Four guys three wearing hats one not are jumping at the top of a staircase.
Translated English: Four guys, three wearing hats, one not, are leaping on top of a staircase.
